# 📈 Corporación Favorita Grocery Sales Forecasting

## Objetivo del Proyecto

Corporación Favorita es una cadena de supermercados de Ecuador que busca predecir las ventas futuras de miles de productos distribuidos en múltiples tiendas.

El objetivo de este proyecto es construir un sistema de análisis y predicción que permita:

- Comprender el comportamiento histórico de las ventas.
- Identificar patrones de demanda.
- Analizar el impacto de promociones, festivos y precios del petróleo.
- Generar modelos predictivos para mejorar la toma de decisiones comerciales.

## Preguntas de Negocio

Durante este proyecto intentaremos responder:

1. ¿Qué productos generan más ventas?
2. ¿Qué tiendas presentan mayor actividad comercial?
3. ¿Cómo afectan las promociones a las ventas?
4. ¿Existen patrones estacionales?
5. ¿Podemos predecir las ventas futuras con precisión?

# 📂 Importación de Librerías

En esta sección se cargan las herramientas fundamentales para el proyecto.

### Librerías utilizadas

- Pandas → Manipulación de datos.
- NumPy → Operaciones numéricas.
- Matplotlib → Visualización.
- Seaborn → Gráficos estadísticos.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# 📥 Carga de los Datasets

El proyecto utiliza múltiples fuentes de información que describen diferentes aspectos del negocio.

## Archivos disponibles

| Dataset | Descripción |
|----------|------------|
| train | Ventas históricas |
| stores | Información de tiendas |
| items | Información de productos |
| transactions | Transacciones por tienda |
| oil | Precio del petróleo |
| holidays | Festivos y eventos |

Cada uno de estos datasets aporta variables relevantes para la construcción del modelo predictivo.

In [2]:
# Definnimos la ruta de los datos
PATH = "../data/raw/"

In [3]:
# Cargamos los datos

train = pd.read_csv(PATH + "train.csv")

stores = pd.read_csv(PATH + "stores.csv")

items = pd.read_csv(PATH + "items.csv")

transactions = pd.read_csv(PATH + "transactions.csv")

oil = pd.read_csv(PATH + "oil.csv")

holidays = pd.read_csv(PATH + "holidays_events.csv")

/var/folders/c5/ylt_kbzs0k5frnlwl06ztm480000gn/T/ipykernel_4352/3639823698.py:3: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(PATH + "train.csv")


# 🔍 Exploración Inicial

Antes de realizar cualquier análisis es importante conocer el tamaño de cada conjunto de datos.

Analizaremos:

- Número de filas.
- Número de columnas.
- Nivel de granularidad.
- Volumen total de información disponible.

In [4]:
# Diccionarios datasets
datasets = {
    "train": train,
    "stores": stores,
    "items": items,
    "transactions": transactions,
    "oil": oil,
    "holidays": holidays,
} 

# Análisis exploratorio de los datos
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(df.info())
    print(df.head())
    print("\n")

train: 125497040 filas, 6 columnas
<class 'pandas.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         str    
 2   store_nbr    int64  
 3   item_nbr     int64  
 4   unit_sales   float64
 5   onpromotion  object 
dtypes: float64(1), int64(3), object(1), str(1)
memory usage: 5.6+ GB
None
   id        date  store_nbr  item_nbr  unit_sales onpromotion
0   0  2013-01-01         25    103665         7.0         NaN
1   1  2013-01-01         25    105574         1.0         NaN
2   2  2013-01-01         25    105575         2.0         NaN
3   3  2013-01-01         25    108079         1.0         NaN
4   4  2013-01-01         25    108701         1.0         NaN


stores: 54 filas, 5 columnas
<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 

# 📊 Comprensión del Dataset Principal `train`

El dataset `train` contiene el historial de ventas.


**Descripción:** Ventas históricas por producto y tienda.

| Columna | Descripción |
|----------|------------|
| id | Identificador único del registro de venta |
| date | Fecha de la venta |
| store_nbr | Identificador de la tienda |
| item_nbr | Identificador del producto |
| unit_sales | Cantidad de unidades vendidas |
| onpromotion | Indica si el producto estaba en promoción (True/False) |

**Producto - Tienda - Fecha**

# 🏪 Exploración de `stores`

El dataset de tiendas contiene información geográfica y operacional.

**Descripción:** Información de las tiendas.

| Columna | Descripción |
|----------|------------|
| store_nbr | Identificador único de la tienda |
| city | Ciudad donde se encuentra la tienda |
| state | Provincia o estado donde se encuentra la tienda |
| type | Tipo de tienda (clasificación interna de la empresa) |
| cluster | Grupo o segmento comercial al que pertenece la tienda |

Preguntas de negocio:

- ¿Cuántas tiendas existen?
- ¿Cómo están distribuidas geográficamente?
- ¿Existen diferencias entre tipos de tienda?

In [ ]:
# ¿Cuántas tiendas existen?
store_counts = stores["store_nbr"].nunique()
print(f"Número de tiendas: {store_counts}\n")
# ¿Cómo están distribuidas geográficamente?
geographical_distribution = stores["city"].value_counts()
print("Distribución geográfica de las tiendas:")
print(f'{geographical_distribution}\n')
# ¿Cuántos  tipos de tienda existen?
type_distribution = stores["type"].value_counts()
print("Distribución por tipo de tienda:")
print(f'{type_distribution}\n')

Número de tiendas: 54

Distribución geográfica de las tiendas:
city
Quito            18
Guayaquil         8
Santo Domingo     3
Cuenca            3
Latacunga         2
Ambato            2
Machala           2
Manta             2
Cayambe           1
Riobamba          1
Ibarra            1
Guaranda          1
Puyo              1
Salinas           1
Daule             1
Babahoyo          1
Quevedo           1
Playas            1
Libertad          1
Loja              1
Esmeraldas        1
El Carmen         1
Name: count, dtype: int64
Distribución por tipo de tienda:
type
D    18
C    15
A     9
B     8
E     4
Name: count, dtype: int64


# 📦 Exploración de `items`

El catálogo de productos contiene información clave para segmentar las ventas.

**Descripción:** Información de los productos.

| Columna | Descripción |
|----------|------------|
| item_nbr | Identificador único del producto |
| family | Familia o categoría del producto |
| class | Clase o subcategoría del producto |
| perishable | Indica si el producto es perecedero (1 = Sí, 0 = No) |

Preguntas de negocio:

- ¿Cuántos productos distintos existen?
- ¿Qué familias tienen mayor presencia?
- ¿Qué porcentaje corresponde a productos perecederos?

In [ ]:
# ¿Cuántos productos distintos existen?
product_counts = items["item_nbr"].nunique()
print(f"Número de productos distintos: {product_counts}\n")
# ¿Qué familias tienen mayor presencia?
family_distribution = items["family"].value_counts()
print("Distribución por familia:")
print(f'{family_distribution}\n') 
# ¿Qué porcentaje corresponde a productos perecederos?
perishable_percentage = items["perishable"].mean() * 100
print(f"Porcentaje de productos perecederos: {perishable_percentage:.2f}%\n")

: 

# Exploración de `transactions`

**Descripción:** Número de transacciones realizadas por tienda y fecha.

| Columna | Descripción |
|----------|------------|
| date | Fecha de la transacción |
| store_nbr | Identificador de la tienda |
| transactions | Número total de transacciones realizadas ese día en la tienda |

In [ ]:
# ¿Cuantas transacciones se realizaron por tienda?
transactions_per_store = transactions.groupby("store_nbr")["transactions"].sum()
print("Número total de transacciones por tienda:")
print(f'{transactions_per_store}\n')
# ¿Cómo varían las transacciones a lo largo del tiempo?
transactions_over_time = transactions.groupby("date")["transactions"].sum()
print("Número total de transacciones a lo largo del tiempo:")
print(f'{transactions_over_time}\n')    

# Dataset: `oil`

**Descripción:** Precio diario del petróleo.

| Columna | Descripción |
|----------|------------|
| date | Fecha del registro |
| dcoilwtico | Precio del petróleo WTI (West Texas Intermediate) en dólares por barril |

 # Dataset: `holidays`

**Descripción:** Festivos y eventos especiales en Ecuador.

| Columna | Descripción |
|----------|------------|
| date | Fecha del evento |
| type | Tipo de evento (Holiday, Event, Additional, Transfer, Bridge, Work Day) |
| locale | Alcance geográfico del evento (National, Regional o Local) |
| locale_name | Nombre de la región o ciudad afectada |
| description | Nombre o descripción del evento |
| transferred | Indica si el festivo fue trasladado a otra fecha (True/False) |

# Resumen de Datasets

| Dataset | Descripción |
|----------|------------|
| train | Ventas históricas por producto y tienda |
| stores | Información de las tiendas |
| items | Información de productos |
| transactions | Transacciones diarias por tienda |
| oil | Precio diario del petróleo WTI |
| holidays | Festivos y eventos especiales de Ecuador |

# 🔗 Integración de Datos

La información se encuentra distribuida en múltiples tablas.

Para realizar un análisis completo es necesario combinar los datasets mediante operaciones equivalentes a SQL JOIN.

Relaciones principales:

train
├── store_nbr → stores
└── item_nbr → items

El objetivo es construir una tabla analítica unificada que contenga:

- Información de ventas.
- Características de las tiendas.
- Características de los productos.

## JOIN 1: Ventas + Tiendas

Incorporamos la información geográfica y operativa de cada tienda al historial de ventas.

In [ ]:
sales = train.merge(
    stores,
    on="store_nbr",
    how="left"
)

sales.head()

## JOIN 2: Ventas + Productos

Añadimos las características de los productos para enriquecer el análisis y permitir futuras segmentaciones.

In [ ]:
sales = sales.merge(
    items,
    on="item_nbr",
    how="left"
)

sales.head()

# 🎯 Conclusiones

Durante esta etapa:

✅ Se cargaron los datasets del proyecto.

✅ Se verificó la calidad estructural de los datos.

✅ Se identificó la unidad de análisis.

✅ Se exploraron las entidades principales del negocio.

✅ Se construyó una tabla analítica integrada para futuras etapas de análisis y modelado.

La siguiente fase consistirá en realizar un Análisis Exploratorio de Datos (EDA) para identificar patrones, tendencias y anomalías en las ventas.